# llama-ultra cloud build (Colab free T4)

Builds `integration/next` (base pin + all ports) with CUDA, runs the new unit tests,
uploads binaries to HuggingFace.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Keep the tab alive
(build ~40-70 min on 2 vCPUs). Safe to re-run any cell: clone pulls instead of
re-cloning, cmake reconfigures in place, rebuilds are incremental.

Binaries are PTX-portable (`75-virtual`): they validate on T4 here and also run
on sm_86 (RTX 3050) via driver JIT. Linux binaries do NOT run on Windows -
the test output pasted back to chat is the deliverable.

In [ ]:
# Cell 1: environment check
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version | head -2
!cmake --version | head -1; command -v ninja >/dev/null && echo HAVE_NINJA || echo NO_NINJA
!nproc; free -g | head -2

In [ ]:
# Cell 2: sources (clone once, pull afterwards)
!if [ -d llama-ultra ]; then cd llama-ultra && git fetch origin && git checkout integration/next && git pull --ff-only; else git clone --depth 1 --branch integration/next https://github.com/TheSyrianDev/llama-ultra.git llama-ultra; fi
!cd llama-ultra && git log --oneline -3 && git status --porcelain | head -3

In [ ]:
# Cell 3: configure (no subshell: GEN must persist) + full CUDA build
!cd llama-ultra && command -v ninja >/dev/null && GEN=Ninja || GEN="Unix Makefiles"; echo "generator=$GEN"; cmake -S . -B build -G "$GEN" -DGGML_CUDA=ON -DGGML_NATIVE=ON -DGGML_CUDA_FA=ON -DCMAKE_CUDA_ARCHITECTURES=75-virtual -DCMAKE_BUILD_TYPE=Release
!cd llama-ultra && time cmake --build build -j$(nproc)

In [ ]:
# Cell 4: port unit tests (synthetic, GPU-enabled, no models needed).
# Paste this cell's full output back to chat.
!cd llama-ultra/build && ./bin/test-llama-archs --test-meta-alloc-failure && ./bin/test-llama-archs --test-sched-copy-name && ./bin/test-llama-archs --test-tied-output-split -s 1 && echo PORT_TESTS_GREEN

In [ ]:
# Cell 5: package + upload to HuggingFace (write token from hf.co/settings/tokens)
!cd llama-ultra && mkdir -p /content/pkg/bin && cp build/bin/llama-server build/bin/llama-bench build/bin/llama-perplexity build/bin/llama-cli /content/pkg/bin/ 2>/dev/null; find build -name 'libggml*.so*' -o -name 'libllama*.so*' | while read f; do cp "$f" /content/pkg/bin/; done; tar czf /content/llama-ultra-t4.tar.gz -C /content/pkg bin && ls -lh /content/llama-ultra-t4.tar.gz
from getpass import getpass
from huggingface_hub import HfApi
tok = getpass('HF write token: ')
api = HfApi()
api.create_repo('OmarBAROOD64/llama-ultra-builds', repo_type='dataset', exist_ok=True, token=tok)
api.upload_file(path_or_fileobj='/content/llama-ultra-t4.tar.gz', path_in_repo='llama-ultra-t4.tar.gz', repo_id='OmarBAROOD64/llama-ultra-builds', repo_type='dataset', token=tok)
print('UPLOADED')
